# 12 - External candidate structure universe (TRAIN + COCONUT + PubChem)

CPU / RAM heavy, GPU not required. **Fully resumable**: every source chunk and every key bucket writes a done-marker on Drive; re-running skips finished work.

1. Stage A - each LOCAL source file (never downloaded by code) is read in chunks -> cheap formula-mass prefilter -> the TRAINING canonicalizer (`standardize_records`: tautomer-canonical InChIKey14 = competition connectivity) -> filters (mass 150-1200, organic, neutral, single component; rejected rows kept with reasons).
2. Stage B - per connectivity-key bucket: `merge.unify` deduplicates by `connectivity_key` (never by raw SMILES / database id / full InChIKey) and keeps provenance.
3. Stage C - global key-sorted `candidate_id`, memory-mapped mass index (`np.searchsorted`), compact formula index.

Scale note: RDKit tautomer canonicalization costs milliseconds per molecule per core. Full PubChem is not realistic in one Colab session; start with COCONUT and a curated PubChem subset (see `configs/v6/pubchem_source.json`).

In [ ]:
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', str(DRIVE_ROOT / 'repo')))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.environment import ensure_packages, set_seeds, system_report
ensure_packages({'rdkit': 'rdkit', 'yaml': 'pyyaml'})
from casmi.workspace.config import load_v2_config, input_path
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '12_colab_candidate_universe'
print(system_report())

In [ ]:
import numpy as np, pandas as pd
from casmi.candidates.universe import UniverseBuildConfig
UCFG = UniverseBuildConfig.from_dict(CFG['universe'])
if UCFG.n_jobs in (None, -1):
    UCFG.n_jobs = os.cpu_count() or 1
OUT = P.candidate_db_dir / 'universe'      # buckets/, variants/, index/, formula_index/, manifests
WORK = P.candidate_db_dir / 'universe_work' # standardized/, rejected/, filtered_out/, _done/
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
print(UCFG)

In [ ]:
# TRAIN structures: the closed-world library keeps its training contract (representative SMILES, mass variants)
from casmi.candidates.filters import train_filter_audit
MV = pd.read_parquet(input_path(CFG, P, 'molecule_mass_variants'))
ST = pd.read_parquet(input_path(CFG, P, 'structure_table'), columns=['smiles', 'connectivity_key', 'molecular_weight', 'formal_charge'])
train_one = MV.sort_values(['connectivity_key', 'mass_variant_id']).drop_duplicates('connectivity_key')
train_one = train_one.merge(ST.dropna(subset=['connectivity_key']).drop_duplicates('connectivity_key')[['connectivity_key', 'formal_charge']],
                            on='connectivity_key', how='left')
print(f'TRAIN: {len(MV):,} mass variants, {MV.connectivity_key.nunique():,} connectivities')
print('filters TRAIN WOULD fail (diagnostic only -- TRAIN is never filtered by default):')
display(train_filter_audit(train_one, UCFG.filters))

In [ ]:
# Stage A - standardize every configured LOCAL source (resumable per chunk)
import time
from casmi.candidates.sources import SourceNotSupplied, load_source_config
from casmi.candidates.universe import standardize_source
STAGE_A = []
for s in CFG['universe']['sources']:
    scfg = load_source_config(REPO_ROOT / s['template'])
    scfg.path = str(DRIVE_ROOT / s['file'])
    if not Path(scfg.path).exists():
        print(f'SKIP {scfg.source}: {scfg.path} not found (obtain it manually and upload it to Drive)')
        continue
    t0 = time.time()
    man = standardize_source(scfg, WORK, UCFG)
    man['seconds'] = time.time() - t0
    STAGE_A.append(man)
STAGE_A = pd.concat(STAGE_A, ignore_index=True) if STAGE_A else pd.DataFrame()
if len(STAGE_A):
    display(STAGE_A.groupby('source')[['n_input', 'n_prefiltered', 'n_rejected', 'n_filtered', 'n_kept']].sum())
else:
    print('no external source processed -> the universe will contain TRAIN only')

In [ ]:
# Stage B - per-bucket dedup by connectivity_key (resumable per bucket)
from tqdm.auto import tqdm
from casmi.candidates.universe import all_buckets, merge_bucket
STD_ROOT = WORK / 'standardized'
BUCKETS = all_buckets(STD_ROOT, MV, UCFG.bucket_prefix_len)
ST_FOR_MERGE = ST.dropna(subset=['connectivity_key'])
BREC = [merge_bucket(b, STD_ROOT, MV, OUT, ST_FOR_MERGE, UCFG) for b in tqdm(BUCKETS, desc='buckets')]
BREC = pd.DataFrame(BREC)
display(BREC.sort_values('n_connectivities', ascending=False).head(10))

In [ ]:
# Stage C - global candidate ids + mass index + formula index
from casmi.candidates.universe import finalize_universe
MANIFEST = finalize_universe(OUT, BUCKETS)
print(json.dumps({k: MANIFEST[k] for k in ('n_candidates', 'n_buckets', 'universe_schema_version')}, indent=2))

In [ ]:
# dedup statistics + source overlap (bucket by bucket: never the whole universe in RAM)
from collections import Counter
overlap, n_records, n_conn = Counter(), 0, 0
for b in BUCKETS:
    t = pd.read_parquet(OUT / 'buckets' / f'bucket={b}.parquet', columns=['candidate_sources', 'n_source_records'])
    overlap.update('+'.join(sorted(s)) for s in t['candidate_sources'])
    n_records += int(t['n_source_records'].sum()); n_conn += len(t)
OVERLAP = pd.DataFrame(sorted(overlap.items(), key=lambda kv: -kv[1]), columns=['sources', 'n_connectivities'])
display(OVERLAP)
DEDUP = {'n_source_records_kept': n_records, 'n_unique_connectivities': n_conn,
         'records_per_connectivity': n_records / max(n_conn, 1)}
print(json.dumps(DEDUP, indent=2))
from casmi.candidates.universe import rejected_summary
REJ = rejected_summary(WORK)
display(REJ.sort_values('n', ascending=False).head(30))

In [ ]:
import matplotlib.pyplot as plt
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.candidates.formula_index import CompactFormulaIndex
MIDX = CandidateMassIndex.load(OUT / 'index')
FIDX = CompactFormulaIndex.load(OUT / 'formula_index')
sample = np.asarray(MIDX.masses[:: max(1, len(MIDX) // 2_000_000)])   # strided sample of the mmap
fc = FIDX.counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(sample, bins=120); axes[0].set_xlabel('exact mass (Da)'); axes[0].set_title(f'mass distribution ({len(MIDX):,} variant rows)')
axes[1].hist(np.log10(fc), bins=60); axes[1].set_xlabel('log10 #candidates per formula'); axes[1].set_title(f'{len(FIDX):,} formulas')
fig.tight_layout()
FIG = P.reports_dir / 'universe'; FIG.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG / 'universe_distributions.png', dpi=120)
top = np.argsort(-fc)[:20]
display(pd.DataFrame({'formula': FIDX.vocab[top], 'n_candidates': fc[top]}))

In [ ]:
from casmi.workspace.artifacts import write_metadata
from casmi.candidates.universe import UNIVERSE_SCHEMA_VERSION, V2_COLUMNS
mem = system_report()
write_metadata(OUT, NOTEBOOK, config={'universe': CFG['universe']}, seed=SEED,
               input_paths={'molecule_mass_variants': input_path(CFG, P, 'molecule_mass_variants'),
                            'structure_table': input_path(CFG, P, 'structure_table'),
                            **{s['template']: DRIVE_ROOT / s['file'] for s in CFG['universe']['sources']}},
               schema_version=UNIVERSE_SCHEMA_VERSION, feature_list=V2_COLUMNS, repo_root=REPO_ROOT,
               extra={'manifest': MANIFEST, 'dedup': DEDUP, 'memory_at_end': mem})
OVERLAP.to_parquet(FIG / 'source_overlap.parquet', index=False)
REJ.to_parquet(FIG / 'rejected_summary.parquet', index=False)
if len(STAGE_A):
    STAGE_A.to_parquet(FIG / 'stage_a_chunks.parquet', index=False)
print('universe saved under', OUT, '| memory:', mem)